# part4.ipynb — Mitigation

**Prerequisite:** Run `part1.ipynb` first.


### i220524 FAST-NUCES Assignment 2 — Responsible & Explainable AI


## Environment setup

Install dependencies (local or Colab), then continue. On Colab: **Runtime → Change runtime type → GPU**.



In [ ]:
"""Install dependencies via `pip install -r requirements.txt` in a fresh environment (e.g. Colab)."""


def ensure_pkg():
    """No-op placeholder; run pip manually or automate here if desired."""
    return None


ensure_pkg()


In [ ]:
"""Imports and global configuration for reproducibility.

Sets USE_TF=0 before HuggingFace loads, and forces trainer_utils.is_tf_available() to False so Trainer.setup never imports TensorFlow (broken TF wheels still report as installed).
"""
import os
os.environ["USE_TF"] = "0"
import re
import random
import zipfile
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.base import clone

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import transformers.trainer_utils as _hf_trainer_utils
_hf_trainer_utils.is_tf_available = lambda: False
from datasets import Dataset

from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ZIP_PATH = os.path.join(os.getcwd(), "jigsaw-unintended-bias-in-toxicity-classification.zip")
TRAIN_CSV_NAME = "train.csv"
ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

BASELINE_DIR = os.path.join(ARTIFACT_DIR, "baseline_distilbert")
POISON_DIR = os.path.join(ARTIFACT_DIR, "poisoned_distilbert")
REWEIGHT_DIR = os.path.join(ARTIFACT_DIR, "reweighted_distilbert")
OVERSAMPLE_DIR = os.path.join(ARTIFACT_DIR, "oversample_distilbert")
ISOTONIC_PATH = os.path.join(ARTIFACT_DIR, "isotonic_calibrator.joblib")

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5

"""Optional: set True only for quick smoke tests (not for submission)."""
FAST_DEBUG = False
if FAST_DEBUG:
    NUM_EPOCHS = 1


## Load data from the zip (stratified 100k train / 20k eval)

We read only the columns needed to limit memory. **Label:** `y = (target >= 0.5).astype(int)`. `train_test_split` returns `(larger_slice, test_size_slice)`; the 20k holdout is the **second** return value.



In [ ]:
"""Load `train.csv` from the competition zip without extracting the full archive to disk."""


def read_train_from_zip(zip_path: str, csv_name: str) -> pd.DataFrame:
    """Read selected columns for memory efficiency and normalize names."""
    usecols = [
        "comment_text",
        "target",
        "black",
        "white",
        "muslim",
        "jewish",
        "homosexual_gay_or_lesbian",
    ]
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(csv_name) as f:
            df = pd.read_csv(f, usecols=usecols)
    df = df.rename(columns={"target": "toxic"})
    df["comment_text"] = df["comment_text"].fillna("").astype(str)
    for c in ["toxic", "black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    df["y"] = (df["toxic"] >= 0.5).astype(int)
    return df


df_all = read_train_from_zip(ZIP_PATH, TRAIN_CSV_NAME)
if FAST_DEBUG:
    eval_n = 2000
    train_n = 5000
    cap = eval_n + train_n + 5000
    df_all, _ = train_test_split(
        df_all, train_size=min(cap, len(df_all)), stratify=df_all["y"], random_state=SEED
    )
else:
    eval_n = 20000
    train_n = 100000

y = df_all["y"].values
train_remaining, eval_df = train_test_split(
    df_all, test_size=eval_n, stratify=y, random_state=SEED
)
y_rem = train_remaining["y"].values
train_df, _discard = train_test_split(
    train_remaining, train_size=train_n, stratify=y_rem, random_state=SEED
)

print("train_df:", train_df.shape, "eval_df:", eval_df.shape)
print("toxic rate train:", train_df["y"].mean(), "eval:", eval_df["y"].mean())


In [ ]:
"""Reload baseline weights and rebuild tokenized datasets (run part1.ipynb first)."""
tokenizer = AutoTokenizer.from_pretrained(BASELINE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(BASELINE_DIR)
model.eval()
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def tokenize_batch(examples, tokenizer_obj, max_length: int):
    """Tokenize comment batches for DistilBERT."""
    return tokenizer_obj(
        examples["comment_text"],
        truncation=True,
        max_length=max_length,
    )


train_ds = Dataset.from_pandas(train_df[["comment_text", "y"]].rename(columns={"y": "labels"}))
eval_ds = Dataset.from_pandas(eval_df[["comment_text", "y"]].rename(columns={"y": "labels"}))
train_ds = train_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)
eval_ds = eval_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)


def compute_metrics(eval_pred):
    """Compute accuracy, macro-F1, binary toxic F1, and AUC when defined."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    pr = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    out = {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_binary": f1_score(labels, preds, pos_label=1),
    }
    try:
        out["auc"] = roc_auc_score(labels, pr)
    except ValueError:
        out["auc"] = float("nan")
    return out


args = TrainingArguments(
    output_dir=os.path.join(ARTIFACT_DIR, "trainer_reload"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    load_best_model_at_end=False,
    report_to=[],
    seed=SEED,
)


def hf_predict_proba_toxic(texts, m, tok, device_obj, max_len):
    """Return toxic-class probabilities for a list of strings."""
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device_obj)
    with torch.no_grad():
        logits = m(**enc).logits
        pr = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
    return pr


texts_eval = eval_df["comment_text"].tolist()
y_true = eval_df["y"].to_numpy()
scores = hf_predict_proba_toxic(texts_eval, model, tokenizer, device, MAX_LENGTH)
CHOSEN_THRESHOLD = 0.5
thr = CHOSEN_THRESHOLD
y_hat = (scores >= thr).astype(int)


In [ ]:
"""Cohort masks used in Part 4 prevalence cell (same definitions as Part 2)."""
cohort_high = eval_df["black"] >= 0.5
cohort_ref = (eval_df["black"] < 0.1) & (eval_df["white"] >= 0.5)


# Part 4 — Mitigation

Three interventions:
1. **Reweighing (aif360)** + `WeightedTrainer`
2. **ThresholdOptimizer (fairlearn, equalized_odds)** + Pareto-style sweep
3. **Oversampling** high-black training comments (`black >= 0.5`) with **3× duplicates**

For reweighing, rows outside the two defined cohorts are treated as **privileged** for weighting (documented engineering simplification).



In [ ]:
"""Reweighing instance weights and retrain DistilBERT with per-example loss scaling."""


def protected_for_fairness(df: pd.DataFrame) -> np.ndarray:
    """Sensitive attribute: 1 = high-black, 0 = reference, 0 = other (privileged default)."""
    hi = df["black"].to_numpy() >= 0.5
    ref = (df["black"].to_numpy() < 0.1) & (df["white"].to_numpy() >= 0.5)
    s = np.zeros(len(df), dtype=float)
    s[hi] = 1.0
    s[ref] = 0.0
    s[~(hi | ref)] = 0.0
    return s


sens_train = protected_for_fairness(train_df)
rw_df = pd.DataFrame({"label": train_df["y"].astype(float), "race": sens_train.astype(float)})
bld_train = BinaryLabelDataset(
    favorable_label=0.0,
    unfavorable_label=1.0,
    df=rw_df,
    label_names=["label"],
    protected_attribute_names=["race"],
    privileged_protected_attributes=[[0.0]],
)
rw = Reweighing(unprivileged_groups=[{"race": 1.0}], privileged_groups=[{"race": 0.0}])
rw_tr = rw.fit_transform(bld_train)
instance_weights = np.asarray(rw_tr.instance_weights).reshape(-1)


class WeightedTrainer(Trainer):
    """HF Trainer variant applying non-uniform instance weights to CE loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Weighted mean cross-entropy."""
        labels = inputs.pop("labels")
        weights = inputs.pop("weights", None)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        if weights is not None:
            w = weights.view(-1).to(loss.device).float()
            loss = (loss * w).sum() / (w.sum() + 1e-8)
        return (loss.mean(), outputs) if return_outputs else loss.mean()


train_rw = Dataset.from_pandas(
    train_df[["comment_text", "y"]].rename(columns={"y": "labels"})
)
train_rw = train_rw.add_column("weights", instance_weights.tolist())
train_rw = train_rw.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)


class WeightedCollator:
    """Pad token batches while preserving per-example loss weights."""

    def __init__(self, tokenizer_obj):
        self.tokenizer = tokenizer_obj

    def __call__(self, features):
        """Return a batch dict including `weights` aligned with `labels`."""
        weights = torch.tensor([float(f["weights"]) for f in features], dtype=torch.float)
        fe = [{k: v for k, v in f.items() if k != "weights"} for f in features]
        batch = self.tokenizer.pad(fe, padding=True, return_tensors="pt")
        batch["weights"] = weights
        return batch


weighted_collator = WeightedCollator(tokenizer)

model_rw = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_rw = clone(args)
args_rw.output_dir = os.path.join(ARTIFACT_DIR, "trainer_reweight")
trainer_rw = WeightedTrainer(
    model=model_rw,
    args=args_rw,
    train_dataset=train_rw,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=weighted_collator,
    compute_metrics=compute_metrics,
)
trainer_rw.train()
trainer_rw.save_model(REWEIGHT_DIR)

model_rw.eval()
model_rw.to(device)
scores_rw = hf_predict_proba_toxic(texts_eval, model_rw, tokenizer, device, MAX_LENGTH)
y_rw_hat = (scores_rw >= thr).astype(int)


In [ ]:
"""fairlearn ThresholdOptimizer on baseline scores (equalized odds, prefit)."""
from sklearn.base import BaseEstimator, ClassifierMixin


class ScoresClassifier(BaseEstimator, ClassifierMixin):
    """Emit fixed probabilities from a precomputed score vector (length must match fit/predict data)."""

    def __init__(self, scores: np.ndarray):
        self.scores = scores.astype(float)
        self.classes_ = np.array([0, 1])

    def fit(self, X, y):
        """No-op fit for prefit ThresholdOptimizer workflows."""
        return self

    def predict_proba(self, X):
        """Return (n,2) probabilities aligned to rows of X."""
        n = len(X)
        s = self.scores[:n]
        return np.column_stack([1.0 - s, s])


sens_eval = protected_for_fairness(eval_df)
mask_two = (sens_eval == 0.0) | (sens_eval == 1.0)
X_dummy = np.zeros((int(mask_two.sum()), 1))
y_sub = y_true[mask_two]
s_sub = sens_eval[mask_two]
scores_sub = scores[mask_two]

base_scores = ScoresClassifier(scores_sub)
post = ThresholdOptimizer(
    estimator=base_scores,
    constraints="equalized_odds",
    prefit=True,
    predict_method="predict_proba",
)
post.fit(X_dummy, y_sub, sensitive_features=s_sub)
y_post = post.predict(X_dummy)


In [ ]:
"""Oversample high-black training rows (3× duplicates) and retrain."""
hb_train = train_df[train_df["black"] >= 0.5]
extra = pd.concat([hb_train] * 3, ignore_index=True) if len(hb_train) else hb_train
train_os = pd.concat([train_df, extra], ignore_index=True)

train_os_ds = Dataset.from_pandas(train_os[["comment_text", "y"]].rename(columns={"y": "labels"}))
train_os_ds = train_os_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)

model_os = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_os = clone(args)
args_os.output_dir = os.path.join(ARTIFACT_DIR, "trainer_oversample")
trainer_os = Trainer(
    model=model_os,
    args=args_os,
    train_dataset=train_os_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_os.train()
trainer_os.save_model(OVERSAMPLE_DIR)

model_os.eval()
model_os.to(device)
scores_os = hf_predict_proba_toxic(texts_eval, model_os, tokenizer, device, MAX_LENGTH)
y_os_hat = (scores_os >= thr).astype(int)


In [ ]:
"""Pareto-style sweep: approximate fairness–F1 trade-offs via group threshold grids."""


def eo_diff(y_true_arr, y_hat_arr, sens_arr):
    """Equalized odds difference on rows with sensitive attribute in {0,1}."""
    m = (sens_arr == 0.0) | (sens_arr == 1.0)
    return equalized_odds_difference(
        y_true=y_true_arr[m],
        y_pred=y_hat_arr[m],
        sensitive_features=sens_arr[m],
    )


def spd_diff(y_true_arr, y_hat_arr, sens_arr):
    """Demographic parity difference on the same masked subset."""
    m = (sens_arr == 0.0) | (sens_arr == 1.0)
    return demographic_parity_difference(
        y_true=y_true_arr[m],
        y_pred=y_hat_arr[m],
        sensitive_features=sens_arr[m],
    )


def best_f1_under_eod(y_true_arr, scores_arr, sens_arr, tol: float) -> Tuple[float, float, np.ndarray]:
    """Greedy grid over group-specific thresholds subject to an EOD tolerance."""
    g0 = sens_arr == 0.0
    g1 = sens_arr == 1.0
    thr0_list = np.linspace(0.05, 0.95, 25)
    thr1_list = np.linspace(0.05, 0.95, 25)
    best_f1 = -1.0
    best_eod = float("nan")
    best_hat = None
    for t0 in thr0_list:
        for t1 in thr1_list:
            y_hat = np.zeros_like(y_true_arr)
            y_hat[g0] = (scores_arr[g0] >= t0).astype(int)
            y_hat[g1] = (scores_arr[g1] >= t1).astype(int)
            eod = abs(eo_diff(y_true_arr, y_hat, sens_arr))
            if eod <= tol + 1e-6:
                f1 = f1_score(y_true_arr, y_hat, average="macro")
                if f1 > best_f1:
                    best_f1 = f1
                    best_eod = eod
                    best_hat = y_hat
    if best_hat is None:
        y_hat = (scores_arr >= 0.5).astype(int)
        return f1_score(y_true_arr, y_hat, average="macro"), abs(eo_diff(y_true_arr, y_hat, sens_arr)), y_hat
    return best_f1, float(best_eod), best_hat


tols = np.linspace(0.0, 0.3, 11)
pareto_x = []
pareto_y = []
for tol in tols:
    f1, eod, _ = best_f1_under_eod(y_true, scores, sens_eval, float(tol))
    pareto_x.append(eod)
    pareto_y.append(f1)

plt.figure(figsize=(7, 5))
plt.scatter(pareto_x, pareto_y)
for xi, yi, ti in zip(pareto_x, pareto_y, tols):
    plt.annotate(f"{ti:.2f}", (xi, yi), textcoords="offset points", xytext=(4, 4), fontsize=8)
plt.xlabel("|equalized odds difference| (proxy via group thresholds)")
plt.ylabel("overall macro-F1")
plt.title("Pareto-style sweep (post-processing intuition)")
plt.show()


In [ ]:
"""Mitigation comparison table (baseline + three techniques)."""
sens_eval_full = protected_for_fairness(eval_df)


def cohort_metrics_labels(y_true_arr: np.ndarray, y_hat_arr: np.ndarray) -> Dict[str, float]:
    """Group metrics from hard predictions (matches post-processed thresholds)."""
    tn, fp, fn, tp = confusion_matrix(y_true_arr, y_hat_arr).ravel()
    tpr_toxic = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    fpr_nontox = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    fnr_toxic = fn / (tp + fn) if (tp + fn) > 0 else float("nan")
    prec_flag = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    return {
        "TPR_toxic": tpr_toxic,
        "FPR_nontoxic": fpr_nontox,
        "FNR_toxic": fnr_toxic,
        "Precision_flagged": prec_flag,
    }


def table_row(name: str, y_hat_arr: np.ndarray) -> Dict[str, float]:
    """One summary row for the Part 4 comparison table."""
    hi = eval_df["black"].to_numpy() >= 0.5
    ref = (eval_df["black"].to_numpy() < 0.1) & (eval_df["white"].to_numpy() >= 0.5)
    mhi = cohort_metrics_labels(y_true[hi], y_hat_arr[hi])
    mrf = cohort_metrics_labels(y_true[ref], y_hat_arr[ref])
    return {
        "model": name,
        "overall_f1_macro": f1_score(y_true, y_hat_arr, average="macro"),
        "highblack_FPR": mhi["FPR_nontoxic"],
        "reference_FPR": mrf["FPR_nontoxic"],
        "SPD": spd_diff(y_true, y_hat_arr, sens_eval_full),
        "EOD": eo_diff(y_true, y_hat_arr, sens_eval_full),
    }


rows_tbl = [
    table_row("baseline", y_hat),
    table_row("reweight", y_rw_hat),
    table_row("oversample", y_os_hat),
]

y_post_full = y_hat.copy()
y_post_full[mask_two] = y_post
rows_tbl.append(table_row("thresh_opt (subset)", y_post_full))

summary_tbl = pd.DataFrame(rows_tbl)
print(summary_tbl)

best_name = summary_tbl.sort_values("overall_f1_macro", ascending=False).iloc[0]["model"]
print("best overall F1 among rows:", best_name)

name_to_dir = {
    "baseline": BASELINE_DIR,
    "reweight": REWEIGHT_DIR,
    "oversample": OVERSAMPLE_DIR,
    "thresh_opt (subset)": REWEIGHT_DIR,
}
mitigated_rank = summary_tbl[summary_tbl["model"] != "baseline"].sort_values(
    "overall_f1_macro", ascending=False
)
best_mitigated_name = str(mitigated_rank.iloc[0]["model"]) if len(mitigated_rank) else "reweight"
BEST_MITIGATED_DIR = name_to_dir.get(best_mitigated_name, REWEIGHT_DIR)
print("Pipeline backbone (best mitigated row):", best_mitigated_name, "->", BEST_MITIGATED_DIR)


### Part 4 — Demographic parity vs equalized odds (model answer — plug in your `prev` table)

**Demographic parity (DP)** targets **equal predicted-positive rates** across cohorts (who gets flagged overall). **Equalized odds (EO)** targets **equal TPR and FPR** (same errors given the true label).

If **toxic prevalence** differs between cohorts (see your printed **P_toxic**), **DP and EO usually cannot both hold** for one fixed scoring rule: matching **overall flag rates** pushes you toward different **conditional** error rates than matching **TPR/FPR**, unless you add **group-specific thresholds** or accept **big accuracy loss**.

**Edit:** One sentence with your two **P_toxic** values and whether **SPD** vs **EOD** trade off after Technique 2 in your run.



In [ ]:
"""Empirical toxic prevalence by cohort on the evaluation subset."""
prev = pd.DataFrame(
    {
        "cohort": ["high_black", "reference"],
        "P_toxic": [
            float(y_true[cohort_high.values].mean()) if cohort_high.any() else float("nan"),
            float(y_true[cohort_ref.values].mean()) if cohort_ref.any() else float("nan"),
        ],
    }
)
print(prev)
